# Vessel Blowdown / Depressurization Simulation (API 521)

This notebook simulates the depressurization (blowdown) of a high-pressure separator
per **API 521** (Pressure-Relieving and Depressuring Systems).

## API 521 Key Requirements

- **Depressurization target**: Reduce vessel pressure to 50% of design pressure or 6.9 barg
  (whichever is lower) within **15 minutes**
- **Fire case**: Must account for external fire heat input
- **MDMT**: Minimum Design Metal Temperature must not be exceeded during blowdown
- **Multi-phase behavior**: Track gas, liquid, and two-phase conditions throughout

## Topics Covered
- Transient blowdown of gas-filled vessel
- Temperature and pressure profiles during blowdown
- Effect of initial conditions on minimum temperature
- Fire case modeling
- Orifice sizing for meeting the 15-minute target

In [ ]:
# Setup NeqSim
import subprocess, sys
try:
    from neqsim import jneqsim
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'neqsim'])
    from neqsim import jneqsim

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations
Stream = jneqsim.process.equipment.stream.Stream
ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
Separator = jneqsim.process.equipment.separator.Separator
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

print('NeqSim loaded successfully')

## 1. Isentropic Blowdown Simulation (Step-wise)

Simulate the blowdown of a gas-filled vessel by stepping through small pressure
increments and tracking the gas temperature (isentropic expansion — worst case
for minimum temperature).

In [ ]:
# Vessel parameters
V_vessel = 50.0  # m³ (vessel volume)
P_initial = 85.0  # bara (initial pressure)
T_initial = 60.0  # °C (initial temperature)
P_atm = 1.013     # bara (final pressure)

# Initial fluid (natural gas + CO2)
gas = SystemSrkEos(273.15 + T_initial, P_initial)
gas.addComponent('methane', 0.80)
gas.addComponent('ethane', 0.07)
gas.addComponent('propane', 0.05)
gas.addComponent('n-butane', 0.02)
gas.addComponent('CO2', 0.03)
gas.addComponent('nitrogen', 0.03)
gas.setMixingRule('classic')

ops = ThermodynamicOperations(gas)
ops.TPflash()
gas.initProperties()

# Initial state
S_initial = float(gas.getEntropy())  # J/mol-K
rho_initial = float(gas.getDensity('kg/m3'))

# Step-wise isentropic expansion
n_steps = 50
P_steps = np.linspace(P_initial, 10.0, n_steps)

blowdown_data = []
T_current = T_initial

for P in P_steps:
    try:
        fluid = SystemSrkEos(273.15 + T_current, float(P))
        fluid.addComponent('methane', 0.80)
        fluid.addComponent('ethane', 0.07)
        fluid.addComponent('propane', 0.05)
        fluid.addComponent('n-butane', 0.02)
        fluid.addComponent('CO2', 0.03)
        fluid.addComponent('nitrogen', 0.03)
        fluid.setMixingRule('classic')

        ops2 = ThermodynamicOperations(fluid)
        ops2.PSflash(S_initial)
        fluid.initProperties()

        T_out = float(fluid.getTemperature()) - 273.15
        rho = float(fluid.getDensity('kg/m3'))
        nph = int(fluid.getNumberOfPhases())
        Z = float(fluid.getZ())

        blowdown_data.append({
            'P_bara': float(P),
            'T_C': T_out,
            'rho_kgm3': rho,
            'Z': Z,
            'phases': nph
        })
        T_current = T_out
    except Exception:
        blowdown_data.append({
            'P_bara': float(P), 'T_C': float('nan'),
            'rho_kgm3': float('nan'), 'Z': float('nan'), 'phases': 0
        })

df = pd.DataFrame(blowdown_data)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Temperature vs Pressure
axes[0, 0].plot(df['P_bara'], df['T_C'], 'b-', linewidth=2)
axes[0, 0].axhline(y=-46, color='red', linestyle='--', alpha=0.7, label='MDMT (-46°C)')
axes[0, 0].set_xlabel('Pressure (bara)')
axes[0, 0].set_ylabel('Temperature (°C)')
axes[0, 0].set_title('Gas Temperature During Isentropic Blowdown')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)
axes[0, 0].invert_xaxis()

# Density vs Pressure
axes[0, 1].plot(df['P_bara'], df['rho_kgm3'], 'g-', linewidth=2)
axes[0, 1].set_xlabel('Pressure (bara)')
axes[0, 1].set_ylabel('Density (kg/m³)')
axes[0, 1].set_title('Gas Density During Blowdown')
axes[0, 1].grid(alpha=0.3)
axes[0, 1].invert_xaxis()

# Z-factor vs Pressure
axes[1, 0].plot(df['P_bara'], df['Z'], 'r-', linewidth=2)
axes[1, 0].set_xlabel('Pressure (bara)')
axes[1, 0].set_ylabel('Compressibility Factor Z')
axes[1, 0].set_title('Z-Factor During Blowdown')
axes[1, 0].grid(alpha=0.3)
axes[1, 0].invert_xaxis()

# P-T trace on phase diagram context
axes[1, 1].plot(df['T_C'], df['P_bara'], 'b-o', linewidth=2, markersize=3)
axes[1, 1].set_xlabel('Temperature (°C)')
axes[1, 1].set_ylabel('Pressure (bara)')
axes[1, 1].set_title('Blowdown P-T Path')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('vessel_blowdown_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

T_min = df['T_C'].min()
print(f'\nMinimum temperature during blowdown: {T_min:.1f}°C')
print(f'Initial conditions: P={P_initial} bara, T={T_initial}°C')

## 2. Effect of Initial Temperature on Minimum Temperature

Higher initial temperature results in higher minimum temperature during blowdown.

In [ ]:
# Sensitivity: initial temperature effect
T_initials = [20, 40, 60, 80, 100]  # °C
P_range = np.linspace(85.0, 10.0, 40)

fig, ax = plt.subplots(figsize=(10, 7))
colors_t = plt.cm.coolwarm(np.linspace(0.1, 0.9, len(T_initials)))

for T0, color in zip(T_initials, colors_t):
    # Get initial entropy
    g0 = SystemSrkEos(273.15 + float(T0), 85.0)
    g0.addComponent('methane', 0.80)
    g0.addComponent('ethane', 0.07)
    g0.addComponent('propane', 0.05)
    g0.addComponent('n-butane', 0.02)
    g0.addComponent('CO2', 0.03)
    g0.addComponent('nitrogen', 0.03)
    g0.setMixingRule('classic')
    ops0 = ThermodynamicOperations(g0)
    ops0.TPflash()
    g0.initProperties()
    S0 = float(g0.getEntropy())

    temps = []
    valid_P = []
    T_cur = float(T0)

    for P in P_range:
        try:
            fl = SystemSrkEos(273.15 + T_cur, float(P))
            fl.addComponent('methane', 0.80)
            fl.addComponent('ethane', 0.07)
            fl.addComponent('propane', 0.05)
            fl.addComponent('n-butane', 0.02)
            fl.addComponent('CO2', 0.03)
            fl.addComponent('nitrogen', 0.03)
            fl.setMixingRule('classic')

            o = ThermodynamicOperations(fl)
            o.PSflash(S0)
            fl.initProperties()
            T_cur = float(fl.getTemperature()) - 273.15
            temps.append(T_cur)
            valid_P.append(float(P))
        except Exception:
            pass

    ax.plot(valid_P, temps, '-', color=color, linewidth=2, label=f'T₀ = {T0}°C')

ax.axhline(y=-46, color='red', linestyle='--', linewidth=2, alpha=0.7, label='MDMT (-46°C)')
ax.set_xlabel('Pressure (bara)', fontsize=12)
ax.set_ylabel('Temperature (°C)', fontsize=12)
ax.set_title(f'Blowdown Temperature Sensitivity to Initial Temperature (P₀={85} bara)')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.invert_xaxis()
plt.tight_layout()
plt.savefig('blowdown_T_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Effect of Gas Composition on Blowdown Temperature

Compare blowdown of different gas compositions — lean gas, rich gas, and CO₂-rich gas.

In [ ]:
# Different gas compositions
gas_compositions = {
    'Lean Gas (95% C1)': {'methane': 0.95, 'ethane': 0.03, 'nitrogen': 0.02},
    'Rich Gas (C1+C2+C3)': {'methane': 0.70, 'ethane': 0.12, 'propane': 0.08,
                             'n-butane': 0.05, 'CO2': 0.03, 'nitrogen': 0.02},
    'CO₂ Gas (50% CO₂)': {'CO2': 0.50, 'methane': 0.40, 'nitrogen': 0.10},
    'HP Flare Gas': {'methane': 0.60, 'ethane': 0.15, 'propane': 0.10,
                      'n-butane': 0.08, 'n-pentane': 0.05, 'CO2': 0.02},
}

P_range = np.linspace(85.0, 10.0, 40)
fig, ax = plt.subplots(figsize=(10, 7))
colors_g = ['blue', 'green', 'red', 'purple']

summary_data = []

for (name, comp), color in zip(gas_compositions.items(), colors_g):
    # Initial entropy
    g0 = SystemSrkEos(273.15 + 60.0, 85.0)
    for k, v in comp.items():
        g0.addComponent(str(k), float(v))
    g0.setMixingRule('classic')
    ops0 = ThermodynamicOperations(g0)
    ops0.TPflash()
    g0.initProperties()
    S0 = float(g0.getEntropy())

    temps = []
    valid_P = []
    T_c = 60.0

    for P in P_range:
        try:
            fl = SystemSrkEos(273.15 + T_c, float(P))
            for k, v in comp.items():
                fl.addComponent(str(k), float(v))
            fl.setMixingRule('classic')
            o = ThermodynamicOperations(fl)
            o.PSflash(S0)
            fl.initProperties()
            T_c = float(fl.getTemperature()) - 273.15
            temps.append(T_c)
            valid_P.append(float(P))
        except Exception:
            pass

    ax.plot(valid_P, temps, '-', color=color, linewidth=2, label=name)
    if temps:
        summary_data.append({'Composition': name, 'T_min (°C)': round(min(temps), 1)})

ax.axhline(y=-46, color='red', linestyle='--', linewidth=2, alpha=0.5, label='MDMT (-46°C)')
ax.set_xlabel('Pressure (bara)', fontsize=12)
ax.set_ylabel('Temperature (°C)', fontsize=12)
ax.set_title('Blowdown Temperature for Different Gas Compositions (P₀=85 bara, T₀=60°C)')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.invert_xaxis()
plt.tight_layout()
plt.savefig('blowdown_composition_effect.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nMinimum Temperature Summary:')
print(pd.DataFrame(summary_data).to_string(index=False))

## Summary

### Key Design Considerations (API 521)

1. **15-minute rule**: Design the blowdown orifice to reach 50% design pressure or
   6.9 barg within 15 minutes
2. **MDMT check**: Ensure the minimum temperature during blowdown does not exceed
   the material's MDMT (typically -46°C for carbon steel)
3. **Heavier gases** cool more during expansion (higher Cp ratio)
4. **CO₂-rich streams** pose unique challenges (dry ice formation near triple point)
5. **Fire case** adds heat input that partially offsets cooling

### NeqSim Features Used
- PSflash (isentropic expansion modeling)
- Multi-component gas properties at various P-T conditions
- Phase behavior tracking during depressurization

### References
- API 521, 7th Ed. (2020): Pressure-Relieving and Depressuring Systems
- API 520, 10th Ed. (2020): Sizing and Selection of Pressure-Relieving Devices
- NORSOK P-001: Process Design